In [17]:
import numpy as np
from layers import Softmax

In [19]:
np.random.seed(3)

T, H = 5, 4 # T: 시계열의 길이, H: 은닉 상태의 차원 수
hs = np.random.randn(T, H)
print(hs)
a = np.array([0.8, 0.1, 0.03, 0.05, 0.02]) # 가중치

ar = a.reshape(5, 1).repeat(4, axis=1)
print(ar.shape)

t = hs * ar
print(t.shape)
print(t)

c = np.sum(t, axis=0)
print(c) # 맥락 벡터


[[ 1.78862847  0.43650985  0.09649747 -1.8634927 ]
 [-0.2773882  -0.35475898 -0.08274148 -0.62700068]
 [-0.04381817 -0.47721803 -1.31386475  0.88462238]
 [ 0.88131804  1.70957306  0.05003364 -0.40467741]
 [-0.54535995 -1.54647732  0.98236743 -1.10106763]]
(5, 4)
(5, 4)
[[ 1.43090278e+00  3.49207880e-01  7.71979745e-02 -1.49079416e+00]
 [-2.77388203e-02 -3.54758979e-02 -8.27414815e-03 -6.27000677e-02]
 [-1.31454507e-03 -1.43165409e-02 -3.94159426e-02  2.65386714e-02]
 [ 4.40659021e-02  8.54786532e-02  2.50168211e-03 -2.02338707e-02]
 [-1.09071990e-02 -3.09295463e-02  1.96473487e-02 -2.20213526e-02]]
[ 1.43500812  0.35396455  0.05165691 -1.56921078]


In [15]:
# 미니배치 버전
np.random.seed(3)

N, T, H = 10, 5, 4
hs = np.random.randn(N, T, H) # 미니배치 수: N , 시계열의 길이: T, 은닉 상태의 차원 수: H -> 하나의 미니배치에 10개의 시계열 데이터가 있고 이 시계열 데이터의 길이는 5이며, 각 시계열 데이터의 은닉 상태의 차원 수는 4이다.
print(hs)

a = np.random.randn(N, T)
ar = a.reshape(N, T, 1).repeat(H, axis=2)
print(ar.shape)
print(ar)

t = hs * ar
print(t.shape)

c = np.sum(t, axis=1)
print(c.shape) # 맥락 벡터
print(c)

[[[ 1.78862847e+00  4.36509851e-01  9.64974681e-02 -1.86349270e+00]
  [-2.77388203e-01 -3.54758979e-01 -8.27414815e-02 -6.27000677e-01]
  [-4.38181690e-02 -4.77218030e-01 -1.31386475e+00  8.84622380e-01]
  [ 8.81318042e-01  1.70957306e+00  5.00336422e-02 -4.04677415e-01]
  [-5.45359948e-01 -1.54647732e+00  9.82367434e-01 -1.10106763e+00]]

 [[-1.18504653e+00 -2.05649899e-01  1.48614836e+00  2.36716267e-01]
  [-1.02378514e+00 -7.12993200e-01  6.25244966e-01 -1.60513363e-01]
  [-7.68836350e-01 -2.30030722e-01  7.45056266e-01  1.97611078e+00]
  [-1.24412333e+00 -6.26416911e-01 -8.03766095e-01 -2.41908317e+00]
  [-9.23792022e-01 -1.02387576e+00  1.12397796e+00 -1.31914233e-01]]

 [[-1.62328545e+00  6.46675452e-01 -3.56270759e-01 -1.74314104e+00]
  [-5.96649642e-01 -5.88594380e-01 -8.73882298e-01  2.97138154e-02]
  [-2.24825777e+00 -2.67761865e-01  1.01318344e+00  8.52797841e-01]
  [ 1.10818750e+00  1.11939066e+00  1.48754313e+00 -1.11830068e+00]
  [ 8.45833407e-01 -1.86088953e+00 -6.028851

### WeightSum 계층    

In [16]:
class WeightSum:
  def __init__(self):
    self.params, self.grads = [], []
    self.cache = None
    
  def forward(self, hs, a):  # hs: LSTM 계층의 은닉 상태들, a: 각 시각의 가중치
    N, T, H = hs.shape       # N: 미니배치 크기, T: 시계열 데이터의 수, H: 은닉 상태의 차원 수

    ar = a.reshape(N, T, 1).repeat(H, axis=2)
    t = hs * ar             # 원소 곱 계산, 은닉 상태 hs의 각 행에 가중치 a의 원소를 곱한다.
    c = np.sum(t, axis=1)   # 맥락 벡터

    self.cache = (hs, ar)   # 역전파 계산 시 사용하기 위해 저장
    return c
  
  def backward(self, dc):   # dc: 덧셈 노드로부터 전해지는 미분
    hs, ar = self.cache
    N, T, H = hs.shape

    dt = dc.reshape(N, 1, H).repeat(T, axis=1)  # sum 노드 역전파
    dar = dt * hs
    dhs = dt * ar
    da = np.sum(dar, axis=2)

    return dhs, da

In [21]:
# 가중치 a를 구하는 단계
N, T, H = 10, 5, 4
hs = np.random.randn(N, T, H)
h = np.random.randn(N, H)
hr = h.reshape(N, 1, H).repeat(T, axis=1)

t = hs * hr
print(t.shape)
print(t)

s = np.sum(t, axis=2)
print(s.shape)
print(s)

softmax = Softmax()
a = softmax.forward(s)
print(a.shape)
print(a)

(10, 5, 4)
[[[ 2.98443370e-02  2.44736535e+00  2.15569816e-01  7.45358950e-03]
  [-1.94073838e-01  3.02685454e+00  1.28964630e+00  6.44653200e-03]
  [-6.76294356e-02 -2.15374890e+00 -9.67704035e-01 -7.34444672e-03]
  [-8.95774339e-01  9.54803333e-01 -6.31696929e-01  6.77363007e-04]
  [-5.00391644e-01 -3.47174865e+00 -6.78740548e-02 -5.28918307e-03]]

 [[-3.13239446e-01  2.88295235e-02  9.36391997e-01 -7.50815752e-01]
  [-2.03811750e+00 -1.41580745e+00 -3.16318394e-01 -1.34213010e-02]
  [-3.18666600e-01 -7.72802808e-01  2.88080861e+00 -1.08082825e+00]
  [ 6.81653069e-01  6.49352494e-01 -1.68997201e+00 -1.46939259e+00]
  [-4.02052783e-01  7.08580596e-01  2.28299214e+00  8.14176691e-01]]

 [[ 6.28380598e-01 -6.40801719e-03  5.05831955e-01 -1.09786043e+00]
  [ 5.18473189e-01 -9.29005231e-03  4.08243250e-02 -2.92814393e-01]
  [ 4.01091610e-01  1.00730476e-02 -1.98618253e-01 -3.55274657e-01]
  [ 5.22089143e-01 -2.47124208e-04  1.29285161e-01  1.78190600e+00]
  [-1.75542925e-01 -1.11639081e-0

### AttentionWeight 계층    

In [ ]:
class AttentionWeight:
  def __init__(self):
    self.params, self.grads = [], []
    self.softmax = Softmax()
    self.cache = None
    
  def forward(self,hs,h): # hs: encoder의 출력(각 시각에서 출력된 은닉 상태), h: decoder의 출력
    N, T, H = hs.shape

    # hs와 h의 내적
    hr = h.reshape(N, 1, H).repeat(T, axis=1)  
    t = hs * hr
    # s가 내적 결과 (N, T) N은 미니배치의 수, T는 시계열의 길이, 만약 N =10, T=5라면 s는 (10, 5)의 형상을 가진다.
    s = np.sum(t, axis=2)

    a = self.softmax.forward(s)
    self.cache = (hs, hr)
    return a
  
  def backward(self, da) # da: softmax의 역전파 출력
    hs, hr = self.cache
    N, T, H = hs.shape

    ds = self.softmax.backward(da)
    dt = ds.reshape(N, T, 1).repeat(H, axis=2)
    dhs = dt * hr
    dhr = dt * hs
    dh = np.sum(dhr, axis=1)

    return dhs, dh

### Attention 계층  

In [ ]:
class Attention:
  def __init__(self):
    self.params, self.grads = [], []
    self.attention_weight_layer = AttentionWeight()
    self.weight_sum_layer = WeightSum()
    self.attention_weight = None
    
  def forward(self, hs, h):
    a = self.attention_weight_layer.forward(hs, h) # 가중치 a를 구한다.
    out = self.weight_sum_layer.forward(hs, a)     # 맥락 벡터를 구한다.    
    self.attention_weight = a
    return out
  
  def backward(self, dout):
    dhs0, da = self.weight_sum_layer.backward(dout)
    dhs1, dh = self.attention_weight_layer.backward(da)
    dhs = dhs0 + dhs1
    return dhs, dh